> **Note — notebook surface is moving.** Starting with `v0.4.0`, all notebooks
> in this repository will move to the dedicated
> [`forecastability-examples`](https://github.com/example/forecastability-examples)
> sibling repository. The library itself will keep only deterministic Python
> APIs, scripts under `scripts/`, and recipe pages under `docs/recipes/`.
> See [docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md](../../docs/plan/v0_4_0_examples_repo_split_ultimate_plan.md).

# 10 — Agent-Ready Triage Interpretation (Deterministic Deep Dive)

This notebook is the deterministic **payload/serializer/interpretation deep dive** for
the triage agent boundary. It focuses on how adapter layers transform deterministic
triage outputs into stable, JSON-serialisable structures for downstream consumers.

This notebook is intentionally **not** the end-to-end triage walkthrough consumer.
For the walkthrough surface, use: `notebooks/walkthroughs/03_triage_end_to_end.ipynb`.

## What this notebook covers

1. Running deterministic triage and inspecting raw `TriageResult` evidence
2. Mapping deterministic results into `TriageAgentPayload` structures
3. Inspecting payload schema, warning fields, and deterministic constraints
4. A3 interpretation mapping: deterministic evidence -> deterministic narrative fields
5. Batch mode payload generation for multi-series triage
6. A2 serialisation envelopes (`SerialisedTriageSummary`) for transport-safe payloads
7. Batch envelope serialisation (`serialise_batch`, `serialise_batch_to_json`)
8. Explicit warning and experimental handling in deterministic interpretation outputs


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np

os.environ.setdefault("MPLBACKEND", "Agg")

import matplotlib.pyplot as plt

from forecastability.adapters.agents.triage_agent_interpretation_adapter import (
    TriageAgentInterpretation,
    interpret_payload,
 )
from forecastability.adapters.agents.triage_agent_payload_models import (
    F1ProfilePayload,
    F5LyapunovPayload,
    F7BatchRankPayload,
    TriageAgentPayload,
    f1_profile_payload,
    f2_limits_payload,
    f6_complexity_payload,
    f7_batch_rank_payload,
    triage_agent_payload,
 )
from forecastability.adapters.agents.triage_summary_serializer import (
    SerialisedTriageSummary,
    serialise_batch,
    serialise_batch_to_json,
    serialise_payload,
    serialise_to_json,
 )
from forecastability.triage.batch_models import BatchSeriesRequest, BatchTriageRequest
from forecastability.triage.models import AnalysisGoal, TriageRequest
from forecastability.use_cases.run_batch_triage import run_batch_triage
from forecastability.use_cases.run_triage import run_triage

Path("outputs/figures/agent").mkdir(parents=True, exist_ok=True)
Path("outputs/json").mkdir(parents=True, exist_ok=True)

print("Imports OK")


## Section 1 — Run deterministic triage

We run `run_triage()` on a seasonal synthetic series and inspect the raw `TriageResult`.
The result contains deterministic numerical outputs — no narrative, no agent logic.

In [ ]:
rng = np.random.default_rng(42)
t = np.arange(600)
series = (
    np.sin(2 * np.pi * t / 12)
    + 0.5 * np.sin(2 * np.pi * t / 6)
    + 0.2 * rng.standard_normal(600)
)

request = TriageRequest(series=series, max_lag=20, n_surrogates=99, random_state=42)
result = run_triage(request)

print(f"Blocked: {result.blocked}")
print(f"Readiness: {result.readiness.status}")
if result.interpretation:
    print(f"Forecastability class: {result.interpretation.forecastability_class}")
    print(f"Directness class: {result.interpretation.directness_class}")
    print(f"Modeling regime: {result.interpretation.modeling_regime}")
if result.forecastability_profile:
    print(f"Peak horizon: {result.forecastability_profile.peak_horizon}")
    print(f"Informative horizons: {result.forecastability_profile.informative_horizons}")
if result.complexity_band:
    print(f"Complexity band: {result.complexity_band.complexity_band}")

## Section 2 — Convert to agent-ready payload

The `triage_agent_payload()` factory converts a full `TriageResult` into a
`TriageAgentPayload` — a flat, JSON-serialisable Pydantic model that:

- replaces `numpy` arrays with Python lists
- surfaces warnings explicitly
- marks experimental components
- includes a schema version for forward compatibility

In [ ]:
payload = triage_agent_payload(result, series_id="seasonal_synthetic")

print(f"schema_version: {payload.schema_version}")
print(f"series_id: {payload.series_id}")
print(f"blocked: {payload.blocked}")
print(f"readiness_status: {payload.readiness_status}")
print(f"forecastability_class: {payload.forecastability_class}")
print(f"directness_class: {payload.directness_class}")
print(f"modeling_regime: {payload.modeling_regime}")
print(f"warnings: {payload.warnings}")
print(f"experimental_notes: {payload.experimental_notes}")

In [ ]:
if payload.f1_profile:
    print("=== F1 Profile Payload ===")
    print(f"profile_shape_label: {payload.f1_profile.profile_shape_label}")
    print(f"peak_horizon: {payload.f1_profile.peak_horizon}")
    print(f"informative_horizons: {payload.f1_profile.informative_horizons}")
    print(f"model_now: {payload.f1_profile.model_now}")
    print(f"review_horizons: {payload.f1_profile.review_horizons}")

if payload.f2_limits:
    print("\n=== F2 Limits Payload ===")
    print(f"ceiling_summary: {payload.f2_limits.ceiling_summary}")
    print(f"compression_warning: {payload.f2_limits.compression_warning}")
    print(f"dpi_warning: {payload.f2_limits.dpi_warning}")

if payload.f6_complexity:
    print("\n=== F6 Complexity Payload ===")
    print(f"complexity_band: {payload.f6_complexity.complexity_band}")
    print(f"permutation_entropy: {payload.f6_complexity.permutation_entropy:.4f}")
    print(f"spectral_entropy: {payload.f6_complexity.spectral_entropy:.4f}")

## Section 3 — JSON serialisation

The payload is fully JSON-serialisable — no numpy types, no domain objects.
This is the format an agent, API endpoint, or chat backend would receive.

In [ ]:
payload_dict = json.loads(payload.model_dump_json())
print(json.dumps(payload_dict, indent=2))

## Section 4 — A3 mapping: deterministic payload -> deterministic narrative

The A3 adapter consumes `TriageAgentPayload` and produces a deterministic
`TriageAgentInterpretation` with separate fields for:

- strong-signal narrative (`strong_signal_narrative`)
- caution/reliability narrative (`cautionary_narrative`)
- experimental narrative (`experimental_narrative`)

This preserves the design rule that narrative text is fully explainable by
deterministic payload fields and explicit warnings/experimental flags.

In [ ]:
a3_interpretation: TriageAgentInterpretation = interpret_payload(payload)

print("=== A3 deterministic interpretation ===")
print(f"signal_bucket: {a3_interpretation.signal_bucket}")
print(f"deterministic_summary: {a3_interpretation.deterministic_summary}")
print(f"strong_signal_narrative: {a3_interpretation.strong_signal_narrative}")
print(f"cautionary_narrative: {a3_interpretation.cautionary_narrative}")
print(f"experimental_narrative: {a3_interpretation.experimental_narrative}")
print(f"warnings: {a3_interpretation.warnings}")
print(f"reliability_notes: {a3_interpretation.reliability_notes}")
print(f"experimental_flagged: {a3_interpretation.experimental_flagged}")

print("\n=== Deterministic evidence -> A3 narrative mapping ===")
print(
    f"forecastability_class={payload.forecastability_class} -> "
    f"signal_bucket={a3_interpretation.signal_bucket}"
)
print(
    f"directness_class={payload.directness_class} -> "
    f"strong_signal_narrative={a3_interpretation.strong_signal_narrative is not None}"
)
print(
    f"warnings_count={len(payload.warnings)} -> "
    f"reliability_notes_count={len(a3_interpretation.reliability_notes)}"
)

a3_from_envelope = interpret_payload(serialise_payload(payload))
print(
    "A2 envelope -> A3 mapping preserves deterministic summary:",
    a3_from_envelope.deterministic_summary == a3_interpretation.deterministic_summary,
)

## Section 5 — Batch mode: F7 per-series payloads

When running batch triage, each series result maps to an `F7BatchRankPayload`.

In [ ]:
rng_b = np.random.default_rng(99)
n_b = 400


def _ar1(phi: float, n: int, seed: int) -> list[float]:
    """Generate AR(1) series as a Python list.

    Args:
        phi: Autoregressive coefficient.
        n: Number of observations.
        seed: Random seed.

    Returns:
        List of floats of length n.
    """
    rng_i = np.random.default_rng(seed)
    s = np.zeros(n)
    s[0] = rng_i.standard_normal()
    for i in range(1, n):
        s[i] = phi * s[i - 1] + rng_i.standard_normal()
    return s.tolist()


batch_request = BatchTriageRequest(
    items=[
        BatchSeriesRequest(series_id="high_ar1", series=_ar1(0.92, n_b, 10)),
        BatchSeriesRequest(series_id="medium_ar1", series=_ar1(0.55, n_b, 20)),
        BatchSeriesRequest(series_id="low_ar1", series=_ar1(0.15, n_b, 30)),
        BatchSeriesRequest(series_id="white_noise", series=rng_b.standard_normal(n_b).tolist()),
        BatchSeriesRequest(
            series_id="seasonal",
            series=(
                np.sin(2 * np.pi * np.arange(n_b) / 12)
                + 0.3 * rng_b.standard_normal(n_b)
            ).tolist(),
        ),
    ],
    max_lag=20,
    n_surrogates=99,
    random_state=42,
)

batch_response = run_batch_triage(batch_request)
f7_payloads = [f7_batch_rank_payload(item) for item in batch_response.items]

print(f"{'Rank':<6} {'Series ID':<15} {'Outcome':<10} {'F-Class':<12} {'Complexity':<12} {'Summary'}")
print("-" * 80)
for p in f7_payloads:
    print(
        f"{str(p.batch_rank):<6} {p.series_id:<15} {p.outcome:<10} "
        f"{str(p.forecastability_class or '-'):<12} {str(p.complexity_band or '-'):<12} "
        f"{p.ranking_summary[:40]}"
    )

## Section 6 — Visualisation

A summary figure comparing raw AMI values against the payload's informative-horizon classification.

In [ ]:
from matplotlib.patches import Patch

Path("outputs/figures/agent").mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: AMI profile + payload informative horizons
if result.forecastability_profile is not None and payload.f1_profile is not None:
    profile = result.forecastability_profile
    p1 = payload.f1_profile
    axes[0].bar(
        profile.horizons,
        profile.values.tolist(),
        color=[
            "#2ecc71" if h in p1.informative_horizons else "#e74c3c"
            for h in profile.horizons
        ],
        alpha=0.8,
        edgecolor="white",
        linewidth=0.5,
    )
    axes[0].axhline(
        y=profile.epsilon,
        color="black",
        linestyle="--",
        lw=1.5,
        label=f"ε={profile.epsilon:.3f}",
    )
    axes[0].set_title(
        "F1 Profile: informative (green) vs non-informative (red) horizons",
        fontsize=10,
    )
    axes[0].set_xlabel("Horizon h")
    axes[0].set_ylabel("AMI F(h; I_t)")
    axes[0].legend(fontsize=9)
else:
    axes[0].text(0.5, 0.5, "No profile data", ha="center", va="center",
                 transform=axes[0].transAxes)
    axes[0].set_title("F1 Profile")

# Right: F7 batch ranking coloured by forecastability class
ok_items = [p for p in f7_payloads if p.outcome == "ok" and p.batch_rank is not None]
color_map: dict[str | None, str] = {
    "high": "#27ae60",
    "medium": "#f39c12",
    "low": "#e74c3c",
    None: "#95a5a6",
}
colors = [color_map.get(p.forecastability_class, "#95a5a6") for p in ok_items]
labels = [p.series_id for p in ok_items]
ranks = [p.batch_rank for p in ok_items]

if ok_items:
    axes[1].barh(labels, ranks, color=colors, alpha=0.85)
    axes[1].set_xlabel("Batch rank")
    axes[1].set_title("F7 Batch ranking (coloured by forecastability class)", fontsize=10)
    axes[1].invert_yaxis()
    legend_elements = [
        Patch(facecolor="#27ae60", label="high"),
        Patch(facecolor="#f39c12", label="medium"),
        Patch(facecolor="#e74c3c", label="low"),
    ]
    axes[1].legend(handles=legend_elements, title="F-class", fontsize=8)
else:
    axes[1].text(0.5, 0.5, "No ok items", ha="center", va="center",
                 transform=axes[1].transAxes)
    axes[1].set_title("F7 Batch ranking")

fig.suptitle(
    "A1 Agent Payload Layer — Deterministic Output Visualisation",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.savefig("outputs/figures/agent/a1_payload_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved.")

## Section 7 — A2: Versioned serialisation envelopes

The `triage_summary_serializer` module (A2) wraps any A1 payload in a
`SerialisedTriageSummary` envelope containing:

- `schema_version` — serialisation format version
- `payload_type` — the concrete class name
- `serialised_at` — ISO-8601 UTC timestamp
- `payload` — the full model_dump() dict, JSON-safe (no numpy types)

This envelope is the transport boundary for agents, APIs, and cross-system consumers.


In [ ]:
# Wrap the TriageAgentPayload from Section 2 in an A2 envelope
serialised = serialise_payload(payload)
print(f"schema_version : {serialised.schema_version}")
print(f"payload_type   : {serialised.payload_type}")
print(f"serialised_at  : {serialised.serialised_at}")
print(f"blocked        : {serialised.payload['blocked']}")
print(f"readiness      : {serialised.payload['readiness_status']}")
print(f"fc_class       : {serialised.payload.get('forecastability_class')}")

# Also demonstrate serialise_to_json for a single F1 profile payload
if payload.f1_profile is not None:
    f1_json = serialise_to_json(payload.f1_profile)
    print("\n--- F1ProfilePayload as versioned JSON ---")
    import json as _json
    f1_envelope = _json.loads(f1_json)
    print(f"payload_type   : {f1_envelope['payload_type']}")
    print(f"payload fields : {list(f1_envelope['payload'].keys())}")

    # Save F1 envelope
    out = Path("outputs/json/nb10_a2_f1_envelope.json")
    out.write_text(f1_json, encoding="utf-8")
    print(f"\nSaved to {out}")


## Section 8 — A2: Batch envelope serialisation

`serialise_batch()` and `serialise_batch_to_json()` wrap a list of payloads
in one call — useful when returning multiple diagnostic results to an agent
or API handler.


In [ ]:
# Collect individual diagnostic payloads from Section 2 result
individual_payloads: list[object] = []
if payload.f1_profile is not None:
    individual_payloads.append(payload.f1_profile)
if payload.f2_limits is not None:
    individual_payloads.append(payload.f2_limits)
if payload.f6_complexity is not None:
    individual_payloads.append(payload.f6_complexity)

batch_envelopes = serialise_batch(individual_payloads)
print(f"Serialised {len(batch_envelopes)} diagnostic payloads")
for env in batch_envelopes:
    print(f"  {env.payload_type:<30} schema_version={env.schema_version}")

# Save full batch as a JSON array
batch_json_path = Path("outputs/json/nb10_a2_diagnostic_batch.json")
batch_json_path.write_text(
    serialise_batch_to_json(individual_payloads), encoding="utf-8"
)
print(f"\nBatch JSON saved to {batch_json_path}")


## Section 9 — A3 explicit warning and experimental handling

This section demonstrates explicit caution handling in A3. We construct a
notebook-only payload variant that includes additional warning and experimental
flags so the separation between signal narrative and reliability narrative is clear.

The underlying domain science is unchanged; this is adapter-boundary behavior only.

In [ ]:
warning_experimental_payload = payload.model_copy(
    update={
        "readiness_status": "warning",
        "warnings": payload.warnings + ["Synthetic notebook caution: inspect stability."],
        "experimental_notes": payload.experimental_notes + [
            "Synthetic notebook experimental note: adapter demonstration.",
        ],
        "f5_lyapunov": F5LyapunovPayload(
            lyapunov_estimate=0.03,
            lyapunov_interpretation="Weakly chaotic.",
            lyapunov_warning="Treat LLE as experimental.",
            experimental_flag_required=True,
            is_experimental=True,
        ),
    }
)

a3_warning_experimental = interpret_payload(warning_experimental_payload)

print("signal_bucket:", a3_warning_experimental.signal_bucket)
print("warnings:", a3_warning_experimental.warnings)
print("reliability_notes:", a3_warning_experimental.reliability_notes)
print("experimental_flagged:", a3_warning_experimental.experimental_flagged)
print("experimental_notes:", a3_warning_experimental.experimental_notes)
print("cautionary_narrative:", a3_warning_experimental.cautionary_narrative)
print("experimental_narrative:", a3_warning_experimental.experimental_narrative)

## Summary

This notebook demonstrated the A1 + A2 + A3 agent adapter layers:

| Component | Purpose |
|---|---|
| `TriageAgentPayload` | Top-level JSON-serialisable payload from `TriageResult` |
| `F1ProfilePayload` | Profile shape, informative horizons, recommendations |
| `F2LimitsPayload` | Theoretical ceiling, compression and DPI warnings |
| `F6ComplexityPayload` | Permutation entropy, spectral entropy, complexity band |
| `F7BatchRankPayload` | Per-series batch rank, outcome, diagnostic vector |
| `triage_agent_payload()` | Factory: `TriageResult` -> `TriageAgentPayload` |
| `SerialisedTriageSummary` | Versioned envelope: payload_type, schema_version, serialised_at, payload dict |
| `serialise_payload()` | Factory: any A1 payload -> `SerialisedTriageSummary` |
| `serialise_batch()` | Factory: list of A1 payloads -> `list[SerialisedTriageSummary]` |
| `TriageAgentInterpretation` | A3 deterministic narrative payload with explicit caution fields |
| `interpret_payload()` | Factory: A1 payload or A2 envelope -> `TriageAgentInterpretation` |

**Key design rule**: The adapter adds no new science. All fields are derived
from deterministic domain results. The narrative is built from fields and
explicit warning/experimental flags, not from an LLM.

The A3 layer keeps **strong-signal**, **cautionary**, and **experimental**
narratives in separate fields, so downstream agent consumers can distinguish
high-confidence signal claims from lower-reliability diagnostic context.